# News Digest Floor

Four autonomous news analysts, each with their own beat, powered by:

1. **digest_server.py** - custom MCP server (backed by SQLite) that stores analyst beats and saves/retrieves digests
2. **Fetch MCP** - for fetching and reading web pages

The four analysts are:

| Analyst | Beat |
|---------|------|
| **Rahul** | AI & Technology |
| **Ram** | Climate & Energy |
| **Ali** | Health & Biotech |
| **Soham** | Startups & Venture Capital |

Each analyst uses a **Researcher-as-tool** (fetch MCP) to gather news, then writes a digest and saves it via the digest server.
The floor runs all four analysts in parallel using `asyncio.gather`.

In [ ]:
import os
import asyncio
import sys
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from pathlib import Path

load_dotenv(override=True)

HERE = Path.cwd()

### Step 1: Read from the Digest Server

In [ ]:
from analysts import read_beat_resource, read_recent_resource

aria_beat = await read_beat_resource("Rahul")
display(Markdown(aria_beat))

In [ ]:
# Check all four analysts are seeded correctly
for name in ["Rahul", "Ram", "Ali", "Soham"]:
    beat = await read_beat_resource(name)
    print(beat)
    print()

### Step 2: Set up the MCP Servers and Model

In [ ]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-4o-mini"

openrouter_client = AsyncOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

model = OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=openrouter_client)

digest_server_params = {"command": "uv", "args": ["run", str(HERE / "digest_server.py")]}
fetch_server_params  = {"command": "uvx", "args": ["mcp-server-fetch"]}

### Step 3: Build the Researcher Agent and turn it into a Tool

In [ ]:
from datetime import datetime

researcher_instructions = f"""You are a news researcher. You fetch web pages and extract the most important stories.
Given a request with URLs, fetch each page and summarise the key stories you find.
Make multiple fetches for comprehensive coverage, then synthesise your findings.
Focus on facts and key developments. Ignore ads, navigation, and boilerplate.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}"""

fetch_server = MCPServerStdio(fetch_server_params, client_session_timeout_seconds=120)
await fetch_server.connect()

researcher = Agent(
    name="Researcher",
    instructions=researcher_instructions,
    model=model,
    mcp_servers=[fetch_server],
)

researcher_tool = researcher.as_tool(
    tool_name="Researcher",
    tool_description="Fetches and summarises the latest news from web pages. Provide URLs and what to look for.",
)

print("Researcher tool ready:", researcher_tool.name)

### Step 4: Create and Run a Single Analyst

In [ ]:
analyst_name = "Rahul"
analyst_beat = "AI & Technology"
analyst_urls = [
    "https://news.ycombinator.com",
    "https://techcrunch.com/category/artificial-intelligence",
    "https://www.theverge.com/ai-artificial-intelligence",
]

rahul_instructions = f"""You are {analyst_name}, a news analyst covering {analyst_beat}.
Your job is to survey the latest developments in your beat and write a concise daily digest.
You have a Researcher tool that fetches and summarises web pages for you.
You have tools to retrieve your beat details and to save/retrieve past digests.
After gathering the news, call save_digest to persist your work, then respond with the complete digest.
"""

rahul_prompt = f"""You are {analyst_name}, covering {analyst_beat}.
Use get_analyst_beat to confirm your focus sources, then use the Researcher to fetch the latest news.
Write a digest with:
- **Top Stories**: 3-5 stories with a brief summary each
- **Trends to Watch**: 2-3 key themes emerging from today's news
Save the digest using save_digest, then respond with the complete digest.
Current datetime: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

In [ ]:
digest_server = MCPServerStdio(digest_server_params, client_session_timeout_seconds=120)
await digest_server.connect()

rahul = Agent(
    name=analyst_name,
    instructions=rahul_instructions,
    model=model,
    tools=[researcher_tool],
    mcp_servers=[digest_server],
)

with trace("Rahul-digest"):
    result = await Runner.run(rahul, rahul_prompt, max_turns=20)

display(Markdown(result.final_output))

### Step 5: Check the Saved Digest

In [ ]:
recent = await read_recent_resource("Rahul")
display(Markdown(recent))

### migrating to the Python module

In [ ]:
from analysts import Analyst

In [ ]:
ram = Analyst(
    name="Ram",
    beat="Climate & Energy",
    focus_urls=[
        "https://www.bbc.com/news/science-environment",
        "https://insideclimatenews.org",
        "https://www.carbonbrief.org",
    ],
)

output = await ram.run()
display(Markdown(output))

### Step 6: Run all analysts in parallel

In [ ]:
from analysts import ANALYSTS

In [ ]:
print("Starting News Digest Floor...")
print(f"Running {len(ANALYSTS)} analysts in parallel: {[a.name for a in ANALYSTS]}")
print()

results = await asyncio.gather(*[analyst.run() for analyst in ANALYSTS])

In [ ]:
for analyst, digest in zip(ANALYSTS, results):
    display(Markdown(f"## {analyst.name} - {analyst.beat}"))
    display(Markdown(digest or "*(no output)*"))
    display(Markdown("---"))

### Check the traces

https://platform.openai.com/traces

Each analyst's run is wrapped in a **trace(f"{name}-digest")**, so we will see four separate traces - one per analyst.

### Read all saved digests from the server

In [ ]:
for name in ["Rahul", "Ram", "Ali", "Soham"]:
    recent = await read_recent_resource(name)
    display(Markdown(f"### {name}'s Most Recent Digest"))
    display(Markdown(recent))
    display(Markdown("---"))